# Flipkart Customer Support Data Analysis

**Project Type:** Exploratory Data Analysis (EDA)  
**Contribution:** Individual  
**Student:** Aditya Kumar Yadav

## Project Summary
This project analyzes Flipkart customer-support data to understand customer issues, support channels, agent workload, response behaviour, and customer satisfaction. The notebook follows data loading, quality checks, preprocessing, UBM analysis (Univariate, Bivariate and Multivariate), 20 meaningful visualizations, business insights, and conclusion.

## Problem Statement
Customer-support operations generate large volumes of interaction data. Systematic analysis can help identify common issues, understand channel and shift workload, examine response behaviour, and evaluate customer satisfaction.

## Business Objective
Use exploratory data analysis to identify meaningful patterns in customer-support interactions and translate them into practical operational observations.

## Guidelines Followed
- Clean, commented, top-to-bottom executable code.
- Duplicate and missing-value checks.
- Date/time and text preprocessing.
- UBM framework: Univariate, Bivariate and Multivariate analysis.
- At least 20 meaningful charts.
- Purpose, insight and business relevance documented for the visualization section.

## 1. Know Your Data

In [ ]:
import warnings
warnings.filterwarnings("ignore")
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", None)
print("Libraries imported successfully.")

In [ ]:
DATA_FILE = "Customer_support_data.csv"
df = pd.read_csv(DATA_FILE)
print("Dataset loaded successfully.")
print("Rows:", df.shape[0])
print("Columns:", df.shape[1])

In [ ]:
display(df.head())
print("Dataset shape:", df.shape)
print("Duplicate rows:", df.duplicated().sum())
print("\nMissing values:")
missing = pd.DataFrame({"Missing Values": df.isna().sum(), "Missing %": (df.isna().mean()*100).round(2)})
display(missing[missing["Missing Values"]>0].sort_values("Missing %", ascending=False))
print("\nData types:")
df.info()

### Dataset Understanding
The dataset contains customer-support interaction records. Important analytical fields include support channel, issue category, agent shift, timestamps, order information, customer remarks and CSAT score. The initial quality check identifies missing and duplicate records before analysis.

## 2. Understanding Your Variables

In [ ]:
print("Columns:")
for i, col in enumerate(df.columns, 1): print(f"{i}. {col}")
print("\nDescriptive statistics:")
display(df.describe(include="all").T)

In [ ]:
for col in ["channel_name","category","Agent Shift","CSAT Score","Product_category","Customer_City"]:
    if col in df.columns:
        print(f"\n--- {col} ---")
        display(df[col].value_counts(dropna=False).head(10).to_frame("Count"))

### Variables Description
- **channel_name:** support channel used for the interaction.
- **category:** reported customer issue category.
- **Agent Shift:** agent shift associated with the case.
- **CSAT Score:** customer satisfaction score.
- **Issue_reported at / issue_responded:** timestamps used for response analysis.
- **Customer Remarks:** customer comments when available.
- **Order_id:** order identifier when available.

## 3. Data Wrangling

In [ ]:
df_clean = df.copy().drop_duplicates()

# Convert timestamp fields safely.
for col in ["order_date_time","Issue_reported at","issue_responded","Survey_response_Date"]:
    if col in df_clean.columns:
        df_clean[col] = pd.to_datetime(df_clean[col], errors="coerce", dayfirst=True, format="mixed")

# Clean text values.
for col in df_clean.select_dtypes(include=["object","string"]).columns:
    df_clean[col] = df_clean[col].str.strip()

# Exclude fields with extremely high missingness from the core EDA.
drop_cols=[c for c in ["connected_handling_time","Customer_City","Product_category","Item_price","order_date_time"] if c in df_clean.columns]
df_clean=df_clean.drop(columns=drop_cols)

if "Customer Remarks" in df_clean.columns: df_clean["Customer Remarks"]=df_clean["Customer Remarks"].fillna("No Remarks")
if "Order_id" in df_clean.columns: df_clean["Order_id"]=df_clean["Order_id"].fillna("Not Available")

# Response-time feature.
if {"Issue_reported at","issue_responded"}.issubset(df_clean.columns):
    df_clean["response_time_minutes"]=(df_clean["issue_responded"]-df_clean["Issue_reported at"]).dt.total_seconds()/60
    df_clean.loc[(df_clean["response_time_minutes"]<0)|(df_clean["response_time_minutes"]>10080),"response_time_minutes"]=np.nan

print("Original shape:", df.shape)
print("Cleaned shape:", df_clean.shape)
print("Duplicates remaining:", df_clean.duplicated().sum())
print("Remaining missing values:", int(df_clean.isna().sum().sum()))

### Data Wrangling Summary
Exact duplicates were removed, date/time fields were parsed safely, text fields were standardized, extremely incomplete fields were excluded from the core EDA, and missing remarks/order IDs were made explicit. A response-time feature was derived where both issue timestamps were available.

## 4. Data Visualization, Storytelling & Experimenting with Charts

The visualizations follow the **UBM framework**: Univariate, Bivariate and Multivariate analysis. The five cells below generate **20 charts** in total. Each group includes the chart purpose, data-driven observations and business relevance.

In [ ]:
def top_value(series):
    s=series.dropna()
    return s.value_counts().index[0] if not s.empty else "Not available"

In [ ]:
# Charts 1-4: Univariate Analysis
fig,ax=plt.subplots(2,2,figsize=(16,10))
df_clean["channel_name"].value_counts().head(10).plot.bar(ax=ax[0,0]); ax[0,0].set_title("1. Cases by Support Channel"); ax[0,0].tick_params(axis="x",rotation=45)
df_clean["category"].value_counts().head(10).plot.bar(ax=ax[0,1]); ax[0,1].set_title("2. Top 10 Issue Categories"); ax[0,1].tick_params(axis="x",rotation=45)
df_clean["Agent Shift"].value_counts().plot.bar(ax=ax[1,0]); ax[1,0].set_title("3. Cases by Agent Shift")
df_clean["CSAT Score"].value_counts().sort_index().plot.bar(ax=ax[1,1]); ax[1,1].set_title("4. CSAT Score Distribution")
plt.tight_layout(); plt.show()
print("1:",top_value(df_clean["channel_name"]),"is the most represented support channel.")
print("2:",top_value(df_clean["category"]),"is the most frequent issue category.")
print("3:",top_value(df_clean["Agent Shift"]),"has the highest case volume.")
print("4: CSAT distribution shows how satisfaction responses are spread across scores.")

**Charts 1–4 — Purpose:** Count/bar charts are appropriate for comparing categorical frequencies. **Business relevance:** They establish the basic workload and satisfaction profile and help identify where support demand is concentrated.

In [ ]:
# Charts 5-8: Bivariate Analysis
fig,ax=plt.subplots(2,2,figsize=(16,10))
pd.crosstab(df_clean["channel_name"],df_clean["CSAT Score"]).plot.bar(stacked=True,ax=ax[0,0]); ax[0,0].set_title("5. CSAT Distribution by Channel"); ax[0,0].tick_params(axis="x",rotation=45)
top=df_clean["category"].value_counts().head(8).index; pd.crosstab(df_clean.loc[df_clean.category.isin(top),"category"],df_clean.loc[df_clean.category.isin(top),"Agent Shift"]).plot.bar(ax=ax[0,1]); ax[0,1].set_title("6. Top Issues Across Agent Shifts"); ax[0,1].tick_params(axis="x",rotation=45)
topch=df_clean["channel_name"].value_counts().head(8).index; pd.crosstab(df_clean.loc[df_clean.channel_name.isin(topch),"channel_name"],df_clean.loc[df_clean.channel_name.isin(topch),"category"]).plot.bar(stacked=True,ax=ax[1,0]); ax[1,0].set_title("7. Issue Mix by Support Channel"); ax[1,0].tick_params(axis="x",rotation=45)
pd.crosstab(df_clean["Agent Shift"],df_clean["CSAT Score"]).plot.bar(stacked=True,ax=ax[1,1]); ax[1,1].set_title("8. CSAT Distribution by Agent Shift")
plt.tight_layout(); plt.show()
print("5-8: These comparisons show how satisfaction, issue mix and workload vary across channels and shifts.")

**Charts 5–8 — Purpose:** Crosstabs and stacked bars compare two categorical variables. **Business relevance:** These charts help compare issue mix, satisfaction composition and workload across operational groups.

In [ ]:
# Charts 9-12: Response and Satisfaction Analysis
fig,ax=plt.subplots(2,2,figsize=(16,10))
if "response_time_minutes" in df_clean:
    sns.histplot(df_clean["response_time_minutes"].dropna().clip(upper=180),bins=30,kde=True,ax=ax[0,0]); ax[0,0].set_title("9. Response Time Distribution (0-180 min)")
    sns.boxplot(data=df_clean,x="Agent Shift",y="response_time_minutes",ax=ax[0,1]); ax[0,1].set_ylim(0,180); ax[0,1].set_title("10. Response Time by Agent Shift"); ax[0,1].tick_params(axis="x",rotation=45)
    sns.boxplot(data=df_clean,x="CSAT Score",y="response_time_minutes",ax=ax[1,0]); ax[1,0].set_ylim(0,180); ax[1,0].set_title("11. Response Time vs CSAT")
else:
    for a in ax.flat: a.text(.5,.5,"Response-time data unavailable",ha="center")
df_clean.groupby("channel_name")["CSAT Score"].mean().sort_values(ascending=False).head(10).plot.bar(ax=ax[1,1]); ax[1,1].set_title("12. Average CSAT by Channel"); ax[1,1].tick_params(axis="x",rotation=45)
plt.tight_layout(); plt.show()
print("9-11: Response-time charts describe response behaviour and its variation by shift and satisfaction score.")
print("12: Average CSAT gives a direct channel-level satisfaction comparison.")

**Charts 9–12 — Purpose:** Histograms and boxplots show distributions and group differences. **Business relevance:** Response-time and average-CSAT comparisons can support service-level monitoring and channel review.

In [ ]:
# Charts 13-16: Temporal Analysis
fig,ax=plt.subplots(2,2,figsize=(16,10))
if "Issue_reported at" in df_clean:
    t=df_clean.dropna(subset=["Issue_reported at"]).copy(); t["hour"]=t["Issue_reported at"].dt.hour; t["day"]=t["Issue_reported at"].dt.day_name(); t["month"]=t["Issue_reported at"].dt.to_period("M").astype(str)
    t["hour"].value_counts().sort_index().plot.line(marker="o",ax=ax[0,0]); ax[0,0].set_title("13. Cases by Hour")
    order=["Monday","Tuesday","Wednesday","Thursday","Friday","Saturday","Sunday"]; t["day"].value_counts().reindex(order).plot.bar(ax=ax[0,1]); ax[0,1].set_title("14. Cases by Day of Week"); ax[0,1].tick_params(axis="x",rotation=45)
    t["month"].value_counts().sort_index().plot.line(marker="o",ax=ax[1,0]); ax[1,0].set_title("15. Monthly Case Volume"); ax[1,0].tick_params(axis="x",rotation=45)
    pd.crosstab(t["hour"],t["Agent Shift"]).plot.area(stacked=True,ax=ax[1,1]); ax[1,1].set_title("16. Hour × Agent Shift Workload")
else:
    for a in ax.flat: a.text(.5,.5,"Timestamp data unavailable",ha="center")
plt.tight_layout(); plt.show()
print("13-16: Temporal charts identify demand periods and help compare workload with shift coverage.")

**Charts 13–16 — Purpose:** Line, bar and area charts reveal time-based patterns. **Business relevance:** Demand timing can support staffing and operational planning.

In [ ]:
# Charts 17-20: Multivariate Analysis
fig,ax=plt.subplots(2,2,figsize=(16,10))
topch=df_clean["channel_name"].value_counts().head(8).index; topcat=df_clean["category"].value_counts().head(8).index
h=pd.crosstab(df_clean.loc[df_clean.channel_name.isin(topch)&df_clean.category.isin(topcat),"channel_name"],df_clean.loc[df_clean.channel_name.isin(topch)&df_clean.category.isin(topcat),"category"]); sns.heatmap(h,annot=True,fmt="g",cmap="Blues",ax=ax[0,0]); ax[0,0].set_title("17. Channel × Issue Heatmap")
pd.crosstab(df_clean["Agent Shift"],df_clean["channel_name"]).plot.bar(stacked=True,ax=ax[0,1]); ax[0,1].set_title("18. Agent Shift × Channel")
topcat=df_clean["category"].value_counts().head(10).index; h2=pd.crosstab(df_clean.loc[df_clean.category.isin(topcat),"category"],df_clean.loc[df_clean.category.isin(topcat),"CSAT Score"]); sns.heatmap(h2,annot=True,fmt="g",cmap="Greens",ax=ax[1,0]); ax[1,0].set_title("19. Issue × CSAT Heatmap")
p=df_clean.pivot_table(index="Agent Shift",columns="channel_name",values="CSAT Score",aggfunc="mean"); sns.heatmap(p,annot=True,fmt=".2f",cmap="Oranges",ax=ax[1,1]); ax[1,1].set_title("20. Average CSAT: Shift × Channel")
plt.tight_layout(); plt.show()
print("17-20: Multivariate views reveal interactions among channel, issue, shift and customer satisfaction.")

**Charts 17–20 — Purpose:** Heatmaps and combined categorical views summarize multiple dimensions simultaneously. **Business relevance:** They help identify concentrated issue-channel combinations and satisfaction patterns that are difficult to see in one-variable analysis.

## 5. Key Business Insights

- **Channel management:** Case volume by channel shows where customer demand is concentrated.
- **Issue prioritization:** Frequent issue categories identify recurring customer-support themes.
- **Workforce planning:** Shift and time analysis helps relate support demand to staffing coverage.
- **Customer satisfaction:** CSAT comparisons show how satisfaction varies across channels, shifts and issues.
- **Response monitoring:** Response-time analysis provides an operational view of service responsiveness.
- **Combined analysis:** Multivariate views reveal relationships among channel, issue, shift and satisfaction.

## 6. Conclusion

This project applies a complete EDA workflow to Flipkart customer-support data. It covers data quality, preprocessing, univariate, bivariate and multivariate analysis, 20 visualizations, and business-oriented interpretation. The resulting analysis can support customer-service monitoring, workload planning, issue prioritization and customer-experience improvement.

**Execution requirement:** Keep `Customer_support_data.csv` in the same folder as this notebook and use **Run All** before final submission.